# 0824_lsw_003_structure_comparison

**통합모델 vs 검사유형별 5분리 모델** 구조 비교 노트북입니다. 데이터 전처리(중복 제거, 시간순 6:2:2 분할)는 `main`의 `0824_kimjaehak_005_xgboost_baseline`과 동일하게 맞춰서, 구조 하나만 바뀌었을 때의 차이만 순수하게 비교합니다.

이번 단계에서는 `mapping.json` 기반 피처 마스킹을 적용하지 않습니다(구조와 별개 변수라 섞으면 원인을 구분할 수 없음) — 대신 학습 데이터 기준 상수열만 제거합니다. `mapping.json` 마스킹은 다음 단계(feature selection)에서 다룹니다.

## 1. 설정과 라이브러리

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

EXPERIMENT_ID = "0824_lsw_003_structure_comparison"
RANDOM_STATE = 42
DATA_PATH = Path("../data/raw/dataset.csv")
TARGET = "class"
TIME_COLUMN = "timestamp"
RECORD_ID = "record_id"
MODEL_DIR = Path("../models")

# docs/lsw/project/scope_and_roadmap.md에 정한 임시 총비용 시나리오
COST_SCENARIOS = {"1:10": (1, 10), "1:100": (1, 100)}

assert DATA_PATH.exists(), f"파일을 찾을 수 없습니다: {DATA_PATH.resolve()}"
print("experiment:", EXPERIMENT_ID)

experiment: 0824_lsw_003_structure_comparison


## 2. 원본 데이터 로딩

`0824_kimjaehak_005_xgboost_baseline`과 동일: 첫 번째 열을 `record_id`로 명시합니다.

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]

if source_index_column.startswith("Unnamed:"):
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")

assert raw_df[RECORD_ID].is_unique, "record_id가 고유하지 않습니다."
print("shape:", raw_df.shape)

shape: (440274, 78)


## 3. 완전 중복 행 제거

`record_id`와 `timestamp`를 제외한 나머지 전체 컬럼 기준으로 중복이면 첫 행만 유지합니다(kimjaehak의 baseline과 동일 기준).

In [3]:
dedup_columns = [c for c in raw_df.columns if c not in {RECORD_ID, TIME_COLUMN}]
duplicate_mask = raw_df.duplicated(subset=dedup_columns, keep="first")
duplicate_rows_removed = int(duplicate_mask.sum())

clean_df = raw_df.loc[~duplicate_mask].copy().reset_index(drop=True)

assert not clean_df.duplicated(subset=dedup_columns, keep=False).any()

pd.Series(
    {
        "rows_before": len(raw_df),
        "duplicate_rows_removed": duplicate_rows_removed,
        "rows_after": len(clean_df),
    }
)

rows_before               440274
duplicate_rows_removed     48282
rows_after                391992
dtype: int64

## 4. 피처/타깃 준비

In [4]:
clean_df[TIME_COLUMN] = pd.to_datetime(clean_df[TIME_COLUMN], errors="raise", utc=True)
clean_df = clean_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)

feature_columns_all = [c for c in clean_df.columns if c not in {RECORD_ID, TIME_COLUMN, TARGET}]
print("전체 피처 수:", len(feature_columns_all))
print("inspection_type 포함 여부:", "inspection_type" in feature_columns_all)

전체 피처 수: 75
inspection_type 포함 여부: True


## 5. 시간순 Train/Validation/Test 분할

kimjaehak의 baseline과 동일하게, 누적 행 수 기준 60/80% 지점의 timestamp를 경계로 잡되 같은 timestamp 그룹이 두 세트에 걸치지 않게 합니다. **이 경계는 전체 데이터(모든 검사유형 합산) 기준으로 한 번만 계산**하고, 두 구조(통합/5분리) 모두 이 경계를 그대로 씁니다 — 그래야 두 모델이 정확히 같은 시험 구간에서 평가됩니다.

In [5]:
timestamps = clean_df[TIME_COLUMN]
timestamp_group_sizes = timestamps.value_counts(sort=False).sort_index()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()

train_end_position = int(np.searchsorted(cumulative_rows, len(clean_df) * 0.60, side="left"))
valid_end_position = int(np.searchsorted(cumulative_rows, len(clean_df) * 0.80, side="left"))
train_end_time = timestamp_group_sizes.index[train_end_position]
valid_end_time = timestamp_group_sizes.index[valid_end_position]

train_mask = timestamps <= train_end_time
valid_mask = (timestamps > train_end_time) & (timestamps <= valid_end_time)
test_mask = timestamps > valid_end_time

assert timestamps.loc[train_mask].max() < timestamps.loc[valid_mask].min()
assert timestamps.loc[valid_mask].max() < timestamps.loc[test_mask].min()

pd.DataFrame(
    [
        {"split": name, "rows": int(mask.sum()), "row_ratio_pct": mask.mean() * 100}
        for name, mask in [("train", train_mask), ("validation", valid_mask), ("test", test_mask)]
    ]
).set_index("split")

,rows,row_ratio_pct
split,,
train,235222,60.006837
validation,78374,19.993775
test,78396,19.999388


## 6. 평가 함수 (Slip Rate / Volume Reduction / 총비용 / 임계값 선택)

`0823_lsw_002_baseline`과 같은 정의를 재사용하되, 임계값 탐색은 `0.01` 간격 grid 대신 **실제 관측된 예측확률값을 후보로 쓰는 exact search**로 바꿨습니다 — grid 해상도가 너무 성겨서 놓치는 임계값이 있었던 문제(이전 baseline에서 확인)를 고치기 위함입니다.

In [6]:
def slip_rate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_positive = y_true == 1
    if actual_positive.sum() == 0:
        return 0.0
    fn = ((y_pred == 0) & actual_positive).sum()
    return fn / actual_positive.sum()


def volume_reduction(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_negative = y_true == 0
    if actual_negative.sum() == 0:
        return 0.0
    tn = ((y_pred == 0) & actual_negative).sum()
    return tn / actual_negative.sum()


def total_cost(y_true, y_pred, cost_fp, cost_fn):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return fn * cost_fn + fp * cost_fp


def select_threshold(y_val, proba_val, max_slip_rate=0.01):
    candidates = np.sort(np.unique(proba_val))[::-1]
    for t in candidates:
        y_pred = (proba_val >= t).astype(int)
        if slip_rate(y_val, y_pred) <= max_slip_rate:
            return float(t)
    return 0.0


def evaluate_at_threshold(y_true, proba, threshold):
    y_pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result = {
        "threshold": threshold,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "slip_rate": slip_rate(y_true, y_pred),
        "volume_reduction": volume_reduction(y_true, y_pred),
        "pr_auc": average_precision_score(y_true, proba),
        "roc_auc": roc_auc_score(y_true, proba) if len(np.unique(y_true)) > 1 else float("nan"),
    }
    for name, (cost_fp, cost_fn) in COST_SCENARIOS.items():
        result[f"total_cost_{name}"] = total_cost(y_true, y_pred, cost_fp, cost_fn)
    return result

## 7. Arm A — 통합모델

`inspection_type`을 포함한 전체 피처를 그대로 입력으로 사용하는 단일 XGBoost입니다 (kimjaehak baseline과 같은 구성).

In [7]:
def get_non_constant_columns(candidate_columns, train_frame):
    nunique = train_frame[candidate_columns].nunique()
    return nunique[nunique > 1].index.tolist()


def build_model():
    return XGBClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        objective="binary:logistic",
        eval_metric="logloss",
    )


unified_train_df = clean_df.loc[train_mask]
unified_valid_df = clean_df.loc[valid_mask]
unified_test_df = clean_df.loc[test_mask]

unified_feature_columns = get_non_constant_columns(feature_columns_all, unified_train_df)
print(f"통합모델 피처 수: {len(unified_feature_columns)} / {len(feature_columns_all)}")

unified_model = build_model()
unified_model.fit(unified_train_df[unified_feature_columns], unified_train_df[TARGET])

unified_valid_proba = unified_model.predict_proba(unified_valid_df[unified_feature_columns])[:, 1]
unified_test_proba = unified_model.predict_proba(unified_test_df[unified_feature_columns])[:, 1]

unified_threshold = select_threshold(unified_valid_df[TARGET], unified_valid_proba)
unified_result = evaluate_at_threshold(unified_test_df[TARGET], unified_test_proba, unified_threshold)
unified_result["n_train"] = len(unified_train_df)
unified_result["n_features"] = len(unified_feature_columns)
pd.Series(unified_result, name="unified")

통합모델 피처 수: 69 / 75


threshold           1.785043e-12
tn                  6.500000e+01
fp                  7.608200e+04
fn                  5.000000e+00
tp                  2.244000e+03
slip_rate           2.223210e-03
volume_reduction    8.536121e-04
pr_auc              2.368043e-01
roc_auc             8.463156e-01
total_cost_1:10     7.613200e+04
total_cost_1:100    7.658200e+04
n_train             2.352220e+05
n_features          6.900000e+01
Name: unified, dtype: float64

## 8. Arm B — 검사유형별 5분리 모델

같은 Train/Val/Test 경계 안에서 `inspection_type`별로 subset을 나눠 각각 XGBoost를 학습합니다. `inspection_type` 자체는 각 모델 안에서 상수이므로 상수열 제거 단계에서 자동으로 빠집니다.

In [8]:
split_results = {}
split_models = {}
split_feature_columns = {}
split_test_true = []
split_test_pred = []

for inspection_type in sorted(clean_df["inspection_type"].unique()):
    type_mask = clean_df["inspection_type"] == inspection_type

    type_train_df = clean_df.loc[train_mask & type_mask]
    type_valid_df = clean_df.loc[valid_mask & type_mask]
    type_test_df = clean_df.loc[test_mask & type_mask]

    type_feature_columns = get_non_constant_columns(feature_columns_all, type_train_df)

    model = build_model()
    model.fit(type_train_df[type_feature_columns], type_train_df[TARGET])

    valid_proba = model.predict_proba(type_valid_df[type_feature_columns])[:, 1]
    test_proba = model.predict_proba(type_test_df[type_feature_columns])[:, 1]

    threshold = select_threshold(type_valid_df[TARGET], valid_proba)
    result = evaluate_at_threshold(type_test_df[TARGET], test_proba, threshold)
    result["n_train"] = len(type_train_df)
    result["n_val_pos"] = int((type_valid_df[TARGET] == 1).sum())
    result["n_features"] = len(type_feature_columns)

    split_results[inspection_type] = result
    split_models[inspection_type] = model
    split_feature_columns[inspection_type] = type_feature_columns

    split_test_true.append(type_test_df[TARGET].to_numpy())
    split_test_pred.append((test_proba >= threshold).astype(int))

split_results_df = pd.DataFrame(split_results).T
split_results_df.index.name = "inspection_type"
split_results_df

,threshold,tn,fp,fn,tp,slip_rate,volume_reduction,pr_auc,roc_auc,total_cost_1:10,total_cost_1:100,n_train,n_val_pos,n_features
inspection_type,,,,,,,,,,,,,,
0,0.000006,8857.0,7794.0,22.0,111.0,0.165414,0.531920,0.067265,0.746373,8014.0,9994.0,41961.0,8.0,47.0
1,0.000025,1869.0,8459.0,8.0,776.0,0.010204,0.180964,0.403144,0.875167,8539.0,9259.0,34041.0,239.0,34.0
2,0.000006,3492.0,14958.0,12.0,691.0,0.017070,0.189268,0.356827,0.887697,15078.0,16158.0,74910.0,32.0,24.0
3,0.000007,9700.0,20288.0,43.0,560.0,0.071310,0.323463,0.269568,0.839741,20718.0,24588.0,80792.0,24.0,22.0
4,0.000001,0.0,730.0,0.0,26.0,0.000000,0.000000,0.023284,0.259405,730.0,730.0,3518.0,60.0,24.0


## 9. 구조 비교 — 통합 vs 5분리 (Pooled)

5분리 모델의 test 예측을 전부 이어붙여, 통합모델과 같은 전체 test 모집단 기준으로 직접 비교합니다.

In [9]:
pooled_true = np.concatenate(split_test_true)
pooled_pred = np.concatenate(split_test_pred)

pooled_tn, pooled_fp, pooled_fn, pooled_tp = confusion_matrix(pooled_true, pooled_pred, labels=[0, 1]).ravel()
split_pooled_result = {
    "tn": int(pooled_tn),
    "fp": int(pooled_fp),
    "fn": int(pooled_fn),
    "tp": int(pooled_tp),
    "slip_rate": slip_rate(pooled_true, pooled_pred),
    "volume_reduction": volume_reduction(pooled_true, pooled_pred),
}
for name, (cost_fp, cost_fn) in COST_SCENARIOS.items():
    split_pooled_result[f"total_cost_{name}"] = total_cost(pooled_true, pooled_pred, cost_fp, cost_fn)

comparison_table = pd.DataFrame(
    [
        {
            "구조": "통합모델",
            "Slip Rate": unified_result["slip_rate"],
            "Volume Reduction": unified_result["volume_reduction"],
            "총비용(1:10)": unified_result["total_cost_1:10"],
            "총비용(1:100)": unified_result["total_cost_1:100"],
            "PR-AUC": unified_result["pr_auc"],
        },
        {
            "구조": "5분리모델(pooled)",
            "Slip Rate": split_pooled_result["slip_rate"],
            "Volume Reduction": split_pooled_result["volume_reduction"],
            "총비용(1:10)": split_pooled_result["total_cost_1:10"],
            "총비용(1:100)": split_pooled_result["total_cost_1:100"],
            "PR-AUC": np.nan,
        },
    ]
).set_index("구조")
comparison_table

,Slip Rate,Volume Reduction,총비용(1:10),총비용(1:100),PR-AUC
구조,,,,,
통합모델,0.002223,0.000854,76132,76582,0.236804
5분리모델(pooled),0.037795,0.314103,53079,60729,NaN


## 10. 검사유형별 비교 — 통합모델 vs 5분리모델

통합모델은 전체 Test에 대해 하나의 임계값(`unified_threshold`)만 썼지만, 실제로는 검사유형마다 다르게 작동했을 수 있습니다. 같은 통합모델의 예측을 `inspection_type`별로 나눠서 Slip Rate/Volume Reduction을 다시 계산하고, 5분리모델의 유형별 결과와 나란히 비교합니다.

In [10]:
unified_type_results = {}
for inspection_type in sorted(unified_test_df["inspection_type"].unique()):
    type_row_mask = (unified_test_df["inspection_type"] == inspection_type).to_numpy()
    y_true_type = unified_test_df.loc[type_row_mask, TARGET]
    proba_type = unified_test_proba[type_row_mask]
    y_pred_type = (proba_type >= unified_threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_type, y_pred_type, labels=[0, 1]).ravel()

    unified_type_results[inspection_type] = {
        "n_test": int(type_row_mask.sum()),
        "n_test_pos": int((y_true_type == 1).sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "slip_rate": slip_rate(y_true_type, y_pred_type),
        "volume_reduction": volume_reduction(y_true_type, y_pred_type),
    }

unified_type_df = pd.DataFrame(unified_type_results).T
unified_type_df.index.name = "inspection_type"

type_comparison = pd.DataFrame(
    {
        "통합_Slip Rate": unified_type_df["slip_rate"],
        "통합_Volume Reduction": unified_type_df["volume_reduction"],
        "5분리_Slip Rate": split_results_df["slip_rate"],
        "5분리_Volume Reduction": split_results_df["volume_reduction"],
    }
)
type_comparison

,통합_Slip Rate,통합_Volume Reduction,5분리_Slip Rate,5분리_Volume Reduction
inspection_type,,,,
0,0.000000,0.000000,0.165414,0.531920
1,0.006378,0.006294,0.010204,0.180964
2,0.000000,0.000000,0.017070,0.189268
3,0.000000,0.000000,0.071310,0.323463
4,0.000000,0.000000,0.000000,0.000000


### Confusion Matrix (raw count) — 통합 vs 5분리

비율 대신 실제 건수로 봅니다. 예측 1(불량 의심)로 남는 TP/FP는 원래(모델 도입 전, 전량 수동검사)와 처리 결과가 동일하므로 baseline 대비 실제 변화는 예측 0(자동 정상처리)으로 새로 옮겨간 **TN(이득: 불필요한 검사 감소)**과 **FN(손해: 원래 100% 걸렸을 불량이 유출)** 두 칸에서만 생깁니다.

In [11]:
confusion_comparison = pd.DataFrame(
    {
        "통합_TN(이득)": unified_type_df["tn"],
        "통합_FP": unified_type_df["fp"],
        "통합_FN(손해)": unified_type_df["fn"],
        "통합_TP": unified_type_df["tp"],
        "5분리_TN(이득)": split_results_df["tn"],
        "5분리_FP": split_results_df["fp"],
        "5분리_FN(손해)": split_results_df["fn"],
        "5분리_TP": split_results_df["tp"],
    }
).astype(int)
confusion_comparison.loc["합계(pooled)"] = confusion_comparison.sum()
confusion_comparison

,통합_TN(이득),통합_FP,통합_FN(손해),통합_TP,5분리_TN(이득),5분리_FP,5분리_FN(손해),5분리_TP
inspection_type,,,,,,,,
0,0,16651,0,133,8857,7794,22,111
1,65,10263,5,779,1869,8459,8,776
2,0,18450,0,703,3492,14958,12,691
3,0,29988,0,603,9700,20288,43,560
4,0,730,0,26,0,730,0,26
합계(pooled),65,76082,5,2244,23918,52229,85,2164


## 11. 대안 — Pooled Validation 기준 임계값 (Partial Pooling)

유형별로 각자 임계값을 고르는 방식은 Validation 표본이 적은 유형(특히 type0: 8건)에서 통계적으로 근거가 약하다. "0건 실패를 관찰"해도 표본이 8개뿐이면 진짜 실패율의 95% 상한은 대략 `3/8 ≈ 37.5%`(rule of three)로, 1% 목표를 그 유형 데이터만으로 검증할 자격 자체가 없다(1%를 확인하려면 대략 300건 이상 필요).

대안으로, 이미 학습된 5개 모델(재학습 없음)의 Validation 예측을 전부 합쳐 **하나의 임계값**만 고르고, 5개 모델에 동일하게 적용해본다. 이는 표본이 부족한 그룹이 전체 집단의 정보를 일부 빌려 추정치를 안정화하는 partial pooling이며, "5개 유형이 같은 데이터"라고 가정하는 게 아니라 편향을 조금 감수하고 분산을 크게 줄이는 명시적 트레이드오프다.

In [12]:
pooled_valid_true = []
pooled_valid_proba = []
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    type_valid_df = clean_df.loc[valid_mask & (clean_df["inspection_type"] == inspection_type)]
    model = split_models[inspection_type]
    feature_columns = split_feature_columns[inspection_type]
    proba = model.predict_proba(type_valid_df[feature_columns])[:, 1]
    pooled_valid_true.append(type_valid_df[TARGET].to_numpy())
    pooled_valid_proba.append(proba)

pooled_valid_true = np.concatenate(pooled_valid_true)
pooled_valid_proba = np.concatenate(pooled_valid_proba)

pooled_threshold = select_threshold(pooled_valid_true, pooled_valid_proba)
print(f"pooled validation 표본 수: {len(pooled_valid_true)} (불량 {int((pooled_valid_true == 1).sum())}건)")
print(f"pooled validation 기준 임계값: {pooled_threshold}")

pooled validation 표본 수: 78374 (불량 363건)
pooled validation 기준 임계값: 1.921282546391012e-06


In [13]:
pooled_threshold_results = {}
pooled_thr_test_true = []
pooled_thr_test_pred = []

for inspection_type in sorted(clean_df["inspection_type"].unique()):
    type_test_df = clean_df.loc[test_mask & (clean_df["inspection_type"] == inspection_type)]
    model = split_models[inspection_type]
    feature_columns = split_feature_columns[inspection_type]
    proba = model.predict_proba(type_test_df[feature_columns])[:, 1]

    result = evaluate_at_threshold(type_test_df[TARGET], proba, pooled_threshold)
    pooled_threshold_results[inspection_type] = result

    pooled_thr_test_true.append(type_test_df[TARGET].to_numpy())
    pooled_thr_test_pred.append((proba >= pooled_threshold).astype(int))

pooled_threshold_df = pd.DataFrame(pooled_threshold_results).T
pooled_threshold_df.index.name = "inspection_type"
pooled_threshold_df

,threshold,tn,fp,fn,tp,slip_rate,volume_reduction,pr_auc,roc_auc,total_cost_1:10,total_cost_1:100
inspection_type,,,,,,,,,,,
0,0.000002,6006.0,10645.0,13.0,120.0,0.097744,0.360699,0.067265,0.746373,10775.0,11945.0
1,0.000002,0.0,10328.0,0.0,784.0,0.000000,0.000000,0.403144,0.875167,10328.0,10328.0
2,0.000002,1056.0,17394.0,6.0,697.0,0.008535,0.057236,0.356827,0.887697,17454.0,17994.0
3,0.000002,2890.0,27098.0,17.0,586.0,0.028192,0.096372,0.269568,0.839741,27268.0,28798.0
4,0.000002,3.0,727.0,1.0,25.0,0.038462,0.004110,0.023284,0.259405,737.0,827.0


In [14]:
pooled_thr_true = np.concatenate(pooled_thr_test_true)
pooled_thr_pred = np.concatenate(pooled_thr_test_pred)

pthr_tn, pthr_fp, pthr_fn, pthr_tp = confusion_matrix(pooled_thr_true, pooled_thr_pred, labels=[0, 1]).ravel()
pooled_threshold_overall = {
    "tn": int(pthr_tn),
    "fp": int(pthr_fp),
    "fn": int(pthr_fn),
    "tp": int(pthr_tp),
    "slip_rate": slip_rate(pooled_thr_true, pooled_thr_pred),
    "volume_reduction": volume_reduction(pooled_thr_true, pooled_thr_pred),
}
for name, (cost_fp, cost_fn) in COST_SCENARIOS.items():
    pooled_threshold_overall[f"total_cost_{name}"] = total_cost(pooled_thr_true, pooled_thr_pred, cost_fp, cost_fn)

three_way_comparison = pd.DataFrame(
    [
        {
            "방식": "통합모델",
            "Slip Rate": unified_result["slip_rate"],
            "Volume Reduction": unified_result["volume_reduction"],
            "총비용(1:10)": unified_result["total_cost_1:10"],
            "총비용(1:100)": unified_result["total_cost_1:100"],
        },
        {
            "방식": "5분리 (유형별 임계값)",
            "Slip Rate": split_pooled_result["slip_rate"],
            "Volume Reduction": split_pooled_result["volume_reduction"],
            "총비용(1:10)": split_pooled_result["total_cost_1:10"],
            "총비용(1:100)": split_pooled_result["total_cost_1:100"],
        },
        {
            "방식": "5분리 (pooled 임계값)",
            "Slip Rate": pooled_threshold_overall["slip_rate"],
            "Volume Reduction": pooled_threshold_overall["volume_reduction"],
            "총비용(1:10)": pooled_threshold_overall["total_cost_1:10"],
            "총비용(1:100)": pooled_threshold_overall["total_cost_1:100"],
        },
    ]
).set_index("방식")
three_way_comparison

,Slip Rate,Volume Reduction,총비용(1:10),총비용(1:100)
방식,,,,
통합모델,0.002223,0.000854,76132,76582
5분리 (유형별 임계값),0.037795,0.314103,53079,60729
5분리 (pooled 임계값),0.016452,0.130734,66562,69892


## 12. 참고 — 이론상 최대 허용 Slip Rate (비용 기준 상한 탐색)

지금까지 쓴 Slip Rate ≤1% 목표가 총비용 관점에서 얼마나 "타이트한" 기준인지 참고용으로 확인한다. **이 절의 결과는 이후 실험의 목표를 바꾸는 데 쓰지 않는다** — 어디까지나 1% 기준의 위치를 가늠하기 위한 참고 수치다.

손익분기 유도: 오늘(전량 수동검사) 비용은 `N × 오검비용`(N=실제 false call 수, 전부 검사 중)이다. 모델 적용 비용은 `FP×오검비용 + FN×미검비용`이다. 모델이 더 싸지려면 `FN×미검비용 < TN×오검비용`이어야 한다. **이론상 최선(Volume Reduction 100%, TN=N)**을 가정하면 감당 가능한 FN 상한은 `N/k`(k=비용비율, `오검비용=1`)이고, 이론상 최대 Slip Rate는 `min(1, N/(k·P))`(P=실제 불량 수)이다.

In [15]:
def max_theoretical_slip_rate(n_negative, n_positive, cost_fp, cost_fn):
    return min(1.0, n_negative / ((cost_fn / cost_fp) * n_positive))


cost_ratios_for_reference = {"1:10": (1, 10), "1:20": (1, 20), "1:100": (1, 100)}

theoretical_ceiling_rows = []
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    type_test_df = clean_df.loc[test_mask & (clean_df["inspection_type"] == inspection_type)]
    n_negative = int((type_test_df[TARGET] == 0).sum())
    n_positive = int((type_test_df[TARGET] == 1).sum())
    row = {
        "inspection_type": inspection_type,
        "N(false call)": n_negative,
        "P(defect)": n_positive,
        "N/P": n_negative / n_positive,
    }
    for name, (cost_fp, cost_fn) in cost_ratios_for_reference.items():
        row[f"max_slip_rate({name})"] = max_theoretical_slip_rate(n_negative, n_positive, cost_fp, cost_fn)
    theoretical_ceiling_rows.append(row)

overall_test_df = clean_df.loc[test_mask]
overall_negative = int((overall_test_df[TARGET] == 0).sum())
overall_positive = int((overall_test_df[TARGET] == 1).sum())
overall_row = {
    "inspection_type": "전체",
    "N(false call)": overall_negative,
    "P(defect)": overall_positive,
    "N/P": overall_negative / overall_positive,
}
for name, (cost_fp, cost_fn) in cost_ratios_for_reference.items():
    overall_row[f"max_slip_rate({name})"] = max_theoretical_slip_rate(overall_negative, overall_positive, cost_fp, cost_fn)
theoretical_ceiling_rows.append(overall_row)

theoretical_ceiling_df = pd.DataFrame(theoretical_ceiling_rows).set_index("inspection_type")
theoretical_ceiling_df

,N(false call),P(defect),N/P,max_slip_rate(1:10),max_slip_rate(1:20),max_slip_rate(1:100)
inspection_type,,,,,,
0,16651,133,125.195489,1.0,1.000000,1.000000
1,10328,784,13.173469,1.0,0.658673,0.131735
2,18450,703,26.244666,1.0,1.000000,0.262447
3,29988,603,49.731343,1.0,1.000000,0.497313
4,730,26,28.076923,1.0,1.000000,0.280769
전체,76147,2249,33.858159,1.0,1.000000,0.338582


**해석(참고용)**: 비용비율 1:100(미검이 오검보다 100배 costly)이라는 보수적인 가정에서도, 이론상 최선(Volume Reduction 100%)일 때 감당 가능한 최대 Slip Rate가 유형별로 13~50%대다 — 지금까지 목표로 쓴 1%보다 훨씬 높다. **즉 1% 목표는 순수 총비용 최소화에서 유도된 숫자가 아니라 별도의 안전/품질 기준으로 봐야 한다.** 이 절의 수치는 참고 컨텍스트로만 남기고, 이후 실험의 목표 자체를 이걸로 바꾸지는 않는다.

## 13. 모델 저장

통합모델 1개, 검사유형별 모델 5개를 각각 저장합니다. 모델 파일은 Git에서 제외됩니다.

In [16]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

unified_path = MODEL_DIR / f"{EXPERIMENT_ID}_unified.pkl"
joblib.dump(
    {"model": unified_model, "feature_columns": unified_feature_columns, "threshold": unified_threshold},
    unified_path,
)
print(f"saved: {unified_path}")

for inspection_type, model in split_models.items():
    model_path = MODEL_DIR / f"{EXPERIMENT_ID}_type{inspection_type}.pkl"
    joblib.dump(
        {
            "model": model,
            "feature_columns": split_feature_columns[inspection_type],
            "threshold": split_results[inspection_type]["threshold"],
        },
        model_path,
    )
    print(f"saved: {model_path}")

saved: ..\models\0824_lsw_003_structure_comparison_unified.pkl
saved: ..\models\0824_lsw_003_structure_comparison_type0.pkl
saved: ..\models\0824_lsw_003_structure_comparison_type1.pkl
saved: ..\models\0824_lsw_003_structure_comparison_type2.pkl
saved: ..\models\0824_lsw_003_structure_comparison_type3.pkl
saved: ..\models\0824_lsw_003_structure_comparison_type4.pkl


## 14. 결론 및 다음 단계

### 결과 요약 — 3가지 방식 비교 (전체 pooled)

| 방식 | Slip Rate | Volume Reduction | 총비용(1:10) | 총비용(1:100) |
|---|---:|---:|---:|---:|
| 통합모델 | 0.22% | 0.09% | 76,132 | 76,582 |
| 5분리 (유형별 임계값) | 3.78% | 31.4% | 53,079 | 60,729 |
| 5분리 (pooled 임계값) | 1.65% | 13.1% | 66,562 | 69,892 |

### pooled 임계값의 유형별 결과 (11절)

| inspection_type | threshold | Slip Rate | Volume Reduction |
|---|---:|---:|---:|
| 0 | 1.92e-06 | 9.77% | 36.1% |
| 1 | 1.92e-06 | 0.00% | 0.0% |
| 2 | 1.92e-06 | 0.85% | 5.7% |
| 3 | 1.92e-06 | 2.82% | 9.6% |
| 4 | 1.92e-06 | 3.85% | 0.4% |

### 핵심 관찰

- **통합모델은 "안전해서 좋은 게" 아니라 "모든 유형에서 똑같이 무효해서" 안전하다** (10절). 5개 유형 전부에서 Volume Reduction이 0%에 가깝다.
- **5분리(유형별)는 4개 유형에서 실제 신호를 찾아내지만**, 표본이 적은 유형(0/2/3)에서 Slip Rate가 목표(1%)를 초과한다.
- **Partial pooling(11절)은 전체 Slip Rate를 목표에 가깝게 개선하지만(3.78%→1.65%) type0 개별로는 여전히 9.77%로 목표 초과**, 대신 안정적이던 type1의 Volume Reduction을 18.1%→0%로 희생시킨다. 임계값 방식을 바꾸는 것만으로는 type0의 근본적인 표본 부족 문제를 해결할 수 없다.
- **이론상 최대 허용 Slip Rate(12절, 참고용)를 보면, 비용비율 1:100에서도 유형별 13~50%까지 정당화된다** — 지금 쓰는 1% 목표는 총비용 최소화가 아니라 별도 안전 기준에서 온 숫자임을 확인했다. 이 수치는 참고용으로만 남기고 목표를 바꾸는 데는 쓰지 않는다.

### 다음 단계

- 세 임계값 방식 다 완전하지 않다: 통합(안전하지만 무효) / 유형별(효율적이지만 위험) / pooled(중간이지만 여전히 불충분, 표본 큰 유형을 희생).
- type0처럼 근본적으로 표본이 부족한 유형은 임계값 방식을 바꾸는 것만으로는 해결이 안 된다 — Phase 1(불균형 처리)에서 오버샘플링/가중치로 이 유형의 유효 신호 자체를 늘리는 시도가 필요해 보인다.
- 이 결과를 갖고 4번 노트북(불균형 처리)으로 넘어간다. 임계값 선택은 일단 **유형별 방식**(가장 큰 Volume Reduction)을 기본값으로 유지하되, type0/2/3처럼 불안정한 유형에 불균형 처리가 실제로 도움이 되는지를 핵심 질문으로 삼는다.
- `mapping.json` 마스킹은 다음 feature selection 단계에서 적용한다.
